# Notebook 04b – Anomaly Mining (Nhánh thay thế Bán giám sát)

**Rubric F:** Đề 5 không bắt buộc bán giám sát. Nhánh thay thế: **phát hiện ngày thời tiết bất thường.**

**Mục tiêu:** Phát hiện ngày có điều kiện thời tiết anomaly (cực trị, bất thường so với pattern chung).

**Kỹ thuật (≥2 phương pháp để so sánh):**
1. **Isolation Forest**: phát hiện outlier trong không gian đa chiều (tree-based, hiệu quả với high-dim)
2. **Local Outlier Factor (LOF)**: phát hiện anomaly dựa trên mật độ cục bộ (density-based)
3. **Z-Score (|z|>3σ)**: baseline thống kê đơn giản

**Đánh giá:**
- So sánh tỷ lệ anomaly giữa 3 phương pháp
- Overlap analysis (consensus ≥2 methods)
- Profile ngày bất thường vs bình thường
- Phân tích anomaly theo mùa
- Top ngày bất thường nhất

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from src.data.loader import load_config, load_processed_data
from src.mining.anomaly import WeatherAnomalyDetector
from src.evaluation.report import save_table
from src.visualization import plots

cfg = load_config('../configs/params.yaml')
plots.FIG_DIR = '../outputs/figures'
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

## 1. Chuẩn bị dữ liệu (hourly → daily aggregation)

In [2]:
df = load_processed_data('../data/processed/weather_processed.parquet')
print(f'Loaded: {df.shape}')

detector = WeatherAnomalyDetector(cfg, contamination=0.05)
daily = detector.prepare_daily(df)
X_scaled, feat_cols = detector.get_feature_matrix(daily)
print(f'Daily feature matrix: {X_scaled.shape}')
print(f'Features: {feat_cols[:10]}...')

[loader] Loaded processed data: (95150, 16)
Loaded: (95150, 16)
[anomaly] Daily aggregated: (4018, 31)
Daily feature matrix: (4018, 29)
Features: ['Temperature (C)_mean', 'Temperature (C)_std', 'Temperature (C)_min', 'Temperature (C)_max', 'Apparent Temperature (C)_mean', 'Apparent Temperature (C)_std', 'Apparent Temperature (C)_min', 'Apparent Temperature (C)_max', 'Humidity_mean', 'Humidity_std']...


## 2. Phát hiện Anomaly – 3 phương pháp

### 2.1 Isolation Forest

In [3]:
labels_iso = detector.fit_isolation_forest(X_scaled, daily)
plots.plot_anomaly_timeline(daily, labels_iso.values, 'IsolationForest')

[anomaly] IsolationForest: 201 anomalies (5.0%)
[plots] Saved: ../outputs/figures\anomaly_timeline_isolationforest.png


### 2.2 Local Outlier Factor (LOF)

In [4]:
labels_lof = detector.fit_lof(X_scaled, daily)
plots.plot_anomaly_timeline(daily, labels_lof.values, 'LOF')

[anomaly] LOF: 201 anomalies (5.0%)
[plots] Saved: ../outputs/figures\anomaly_timeline_lof.png


### 2.3 Z-Score Baseline (|z| > 3σ)

In [5]:
labels_z = detector.fit_zscore(X_scaled, daily, threshold=3.0)

[anomaly] Z-Score (σ>3.0): 468 anomalies (11.65%)


## 3. So sánh phương pháp

In [6]:
compare_df = detector.compare_methods()
print(compare_df.to_string(index=False))
save_table(compare_df, 'anomaly_method_comparison', '../outputs/tables')
plots.plot_anomaly_comparison(compare_df)

         Method  N_anomaly  Pct_anomaly(%)
IsolationForest        201            5.00
            LOF        201            5.00
         ZScore        468           11.65
[report] Saved table: ../outputs/tables\anomaly_method_comparison.csv
[plots] Saved: ../outputs/figures\anomaly_method_comparison.png


## 4. Overlap Analysis (Consensus)

Ngày được ≥2 phương pháp đánh dấu anomaly → **consensus anomaly** (tin cậy cao hơn).

In [7]:
overlap = detector.overlap_analysis()
consensus_count = overlap['consensus_anomaly'].sum()
total = len(overlap)
print(f'Consensus anomalies (≥2 methods): {consensus_count} / {total} ngày ({consensus_count/total*100:.2f}%)')

# Phân bố theo số method flagged
flag_dist = overlap['n_methods_flagged'].value_counts().sort_index()
print(f'\nPhân bố số phương pháp flagged:')
for n_methods, count in flag_dist.items():
    print(f'  {n_methods} method(s): {count} ngày ({count/total*100:.1f}%)')

save_table(overlap.reset_index(), 'anomaly_overlap', '../outputs/tables')

[anomaly] Consensus (≥2 methods): 196 anomalies (4.88%)
Consensus anomalies (≥2 methods): 196 / 4018 ngày (4.88%)

Phân bố số phương pháp flagged:
  0 method(s): 3388 ngày (84.3%)
  1 method(s): 434 ngày (10.8%)
  2 method(s): 152 ngày (3.8%)
  3 method(s): 44 ngày (1.1%)
[report] Saved table: ../outputs/tables\anomaly_overlap.csv


'../outputs/tables\\anomaly_overlap.csv'

## 5. Profile: Anomaly vs Normal

In [8]:
profile = detector.profile_anomalies(daily, feat_cols, method='IsolationForest')
print(profile.to_string())
save_table(profile, 'anomaly_profile', '../outputs/tables')
plots.plot_anomaly_profile(profile)

            Count  Temperature (C)_mean  Apparent Temperature (C)_mean  Humidity_mean  Wind Speed (km/h)_mean  Wind Bearing (degrees)_mean  Visibility (km)_mean  Pressure (millibars)_mean
is_anomaly                                                                                                                                                                                 
Normal       3817                12.506                         11.552          0.733                  10.555                      188.686                10.492                   1016.664
Anomaly       201                 0.869                         -2.531          0.774                  15.438                      167.078                 8.259                   1019.758
[report] Saved table: ../outputs/tables\anomaly_profile.csv
[plots] Saved: ../outputs/figures\anomaly_profile_comparison.png


## 6. Anomaly theo mùa

In [9]:
season_anomaly = detector.anomaly_by_season(daily, method='IsolationForest')
print(season_anomaly.to_string())
save_table(season_anomaly, 'anomaly_by_season', '../outputs/tables')
plots.plot_anomaly_by_season(season_anomaly)

        total  n_anomaly  pct_anomaly(%)
Season                                  
Winter    993        137           13.80
Spring   1012         36            3.56
Fall     1001         14            1.40
Summer   1012         14            1.38
[report] Saved table: ../outputs/tables\anomaly_by_season.csv


[plots] Saved: ../outputs/figures\anomaly_by_season.png


## 7. Top ngày bất thường nhất

In [10]:
top_days = detector.get_top_anomaly_days(daily, method='IsolationForest', n=15)
print('Top 15 ngày bất thường nhất (IsolationForest):')
print(top_days.to_string())
save_table(top_days, 'anomaly_top_days', '../outputs/tables')

Top 15 ngày bất thường nhất (IsolationForest):
                Temperature (C)_mean  Apparent Temperature (C)_mean  Humidity_mean  Wind Speed (km/h)_mean  Wind Bearing (degrees)_mean  anomaly_score  Season
Formatted Date                                                                                                                                                
2006-01-23                -11.469444                     -18.281944       0.681667               15.694146                    16.291667      -0.120028  Winter
2012-02-03                -11.202315                     -18.825463       0.680000               17.921313                    19.125000      -0.109356  Winter
2012-02-08                -15.310185                     -17.641435       0.636667                5.948279                   170.500000      -0.095889  Winter
2012-02-10                -15.958796                     -17.815509       0.747917                4.762917                   193.708333      -0.095304  Winter

'../outputs/tables\\anomaly_top_days.csv'

## 8. Insight & Kết luận

In [11]:
print('=== INSIGHT ANOMALY MINING ===')

# So sánh tự động
best_method = compare_df.loc[compare_df['N_anomaly'].idxmin(), 'Method']
print(f'\n1. Phương pháp phát hiện ít anomaly nhất: {best_method}')
print(f'   → Có thể là phương pháp chặt nhất (ít false positive)')

# Mùa nhiều anomaly nhất
worst_season = season_anomaly['pct_anomaly(%)'].idxmax()
worst_pct = season_anomaly.loc[worst_season, 'pct_anomaly(%)']
print(f'\n2. Mùa có nhiều ngày bất thường nhất: {worst_season} ({worst_pct}%)')

# Profile
if len(profile) >= 2:
    for col in profile.columns:
        if col == 'Count': continue
        normal_val = profile.loc['Normal', col]
        anomaly_val = profile.loc['Anomaly', col]
        if abs(anomaly_val - normal_val) / (abs(normal_val) + 1e-8) > 0.3:
            direction = 'cao hơn' if anomaly_val > normal_val else 'thấp hơn'
            print(f'   {col}: Anomaly {direction} Normal ({anomaly_val:.2f} vs {normal_val:.2f})')

print(f'\n3. Consensus (≥2 methods): {consensus_count} ngày ({consensus_count/total*100:.2f}%)')
print(f'   → Đây là những ngày bất thường đáng tin cậy nhất')

print(f'\n4. Ứng dụng: Hệ thống cảnh báo thời tiết cực đoan,')
print(f'   lọc outlier trước khi train mô hình dự báo,')
print(f'   phát hiện sự kiện thời tiết hiếm gặp.')

print(f'\n✓ Anomaly mining hoàn tất.')

=== INSIGHT ANOMALY MINING ===

1. Phương pháp phát hiện ít anomaly nhất: IsolationForest
   → Có thể là phương pháp chặt nhất (ít false positive)

2. Mùa có nhiều ngày bất thường nhất: Winter (13.8%)
   Temperature (C)_mean: Anomaly thấp hơn Normal (0.87 vs 12.51)
   Apparent Temperature (C)_mean: Anomaly thấp hơn Normal (-2.53 vs 11.55)
   Wind Speed (km/h)_mean: Anomaly cao hơn Normal (15.44 vs 10.55)

3. Consensus (≥2 methods): 196 ngày (4.88%)
   → Đây là những ngày bất thường đáng tin cậy nhất

4. Ứng dụng: Hệ thống cảnh báo thời tiết cực đoan,
   lọc outlier trước khi train mô hình dự báo,
   phát hiện sự kiện thời tiết hiếm gặp.

✓ Anomaly mining hoàn tất.
